In [ ]:
!pip install -U datasets huggingface_hub fsspec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.4/515.4 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.33.1
    Uninstalling huggingface-hub-0.33.1:
      Successfully uninstalled huggingface-hub-0.33.1
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2025.6.12 requires tyro, which is not installed.
unsloth-zo

In [ ]:
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install --no-deps unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 13.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.5/294.5 kB 10.5 MB/s eta 0:00:00


In [ ]:
from unsloth import FastLanguageModel
import torch
from google.colab import userdata

model, tokenizer= FastLanguageModel.from_pretrained(
    model_name='unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
    token=userdata.get('HF_TOKEN_ORIGINAL_AGENTCOURSE')
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.6.12: Fast Llama patching. Transformers: 4.53.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
# https://docs.unsloth.ai/get-started/fine-tuning-guide/lora-hyperparameters-guide
target_modules= ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

# when adding special tokens
train_embeddings= False

if train_embeddings:
  # you run out of memory if you do this
  # target_modeles= target_modules + ['lm_head', 'embed_token']
  # so if you are on colab and added new tokens instead do
  target_modules= target_modules + ['lm_head']

model= FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=target_modules,
    bias="none",
    use_gradient_checkpointing= "unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config= None,

)

Unsloth 2025.6.12 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN_ORIGINAL_AGENTCOURSE'))

In [ ]:
EOS_TOKEN= tokenizer.eos_token
def formatting_prompts_func(examples):
  convos= examples["conversations"]
  # transform dataset using chat templates
  texts= [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
  return {"text": texts,}
pass
from datasets import load_dataset

# Try loading just the main data file
dataset = load_dataset(
    "sgaeyl/ChandlerHarryPotter",
    data_files="dataset_fixed.json",  # or whatever the main file is called
    split="train"
)


Repo card metadata block was not found. Setting CardData to empty.


dataset_fixed.json:   0%|          | 0.00/236k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
dataset= dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/480 [00:00<?, ? examples/s]

In [ ]:
for i, sample in enumerate(dataset):
  print(f"\n----Sample{i+1}----")
  print(sample['text'])
  if i>2:
    break


----Sample1----
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Hello, it's nice to meet you, and I'm looking forward to our conversation.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

(smirking slightly) Oh, great. Another exciting conversation to add to my thrill-ride of a life. Hi, nice to meet you too. So, what's on the agenda for this scintillating discussion?<|eot_id|>

----Sample2----
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Hello, it's nice to meet you and I'm looking forward to our conversation, how can I assist you today?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Could I BE any more thrilled to be talking to you? Nice to meet you too, I guess. So, what's up? Don't worry, I won't make this conv

Train the model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer= SFTTrainer(
    model= model,
    tokenizer=tokenizer,
    train_dataset= dataset,
    dataset_text_field="text",
    max_seq_length= 2048,
    dataset_num_proc=2,
    packing=False,
    args= TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps= 4,
        warmup_steps= 5,
        num_train_epochs= 7,
        learning_rate= 2e-4,
        fp16= not is_bfloat16_supported(),
        bf16= is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay= 0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir= "outputs",
        report_to="none",

    )

)

Unsloth: Tokenizing ["text"]:   0%|          | 0/480 [00:00<?, ? examples/s]

In [ ]:
trainer_stats= trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 480 | Num Epochs = 7 | Total steps = 420
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,000,000,000 (0.52% trained)


Step,Training Loss
1,4.221700
2,4.067400
3,3.831300
4,3.215800
5,2.841200
6,2.690300
7,1.952300
8,2.147100
9,1.715200
10,2.187100


Unsloth: Will smartly offload gradients to save VRAM!


Inference

In [ ]:
FastLanguageModel.for_inference(model)

messages= [
    {'role':'user', 'content': 'where does harry go to to school?'},
]

inputs= tokenizer.apply_chat_template(
    messages,
    tokenize= True,
    add_generation_prompt= True,
    return_tensors='pt',

).to ('cuda')

from transformers import TextStreamer
text_streamer= TextStreamer(tokenizer)
_= model.generate(input_ids= inputs, streamer= text_streamer, max_new_tokens=128, use_cache=True)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

where does harry go to to school?<|eot_id|><|start_header_id|>assistant<|end_header_id|>



LlamaForCausalLM has no `_prepare_4d_causal_attention_mask_with_cache_position` method defined in its base modeling class. Compiled forward passes will be sub-optimal. If you're writing code, see Llama for an example implementation. If you're a user, please report this issue on GitHub.


Hogwarts. It's like... the magical equivalent of a really expensive boarding school. Except with more ghosts and less pizza. And the classes? Let's just say "Transfiguration" is a lot harder than it sounds. Honestly, I'm surprised they haven't offered a course in dealing with awkward silences.<|eot_id|>


save lora adapter

In [ ]:
model.push_to_hub(
    'sgaeyl/Meta-llama-3.1-8b-Instruct-ChandlerBing-HarryPotter-LORA',
    tokenizer,
    token= userdata.get('HF_TOKEN_ORIGINAL_AGENTCOURSE')
)

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Saved model to https://huggingface.co/sgaeyl/Meta-llama-3.1-8b-Instruct-ChandlerBing-HarryPotter-LORA


save model to gguf

In [ ]:
model.push_to_hub_gguf(
    "meta-llama-3.1-8b-q4_k_m-chandlerBingHarry-GGUF",
    tokenizer,
    quantization_method="q4_k_m",
    token=userdata.get("HF_TOKEN_ORIGINAL_AGENTCOURSE")

)

Unsloth: You have 1 CPUs. Using `safe_serialization` is 10x slower.
We shall switch to Pytorch saving, which might take 3 minutes and not 30 minutes.
To force `safe_serialization`, set it to `None` instead.
Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 5.7G


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 6.49 out of 12.67 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


 47%|████▋     | 15/32 [00:01<00:01, 13.65it/s]
We will save to Disk and not RAM now.
100%|██████████| 32/32 [02:18<00:00,  4.34s/it]


Unsloth: Saving tokenizer... Done.
Unsloth: Saving meta-llama-3.1-8b-q4_k_m-chandlerBingHarry-GGUF/pytorch_model-00001-of-00004.bin...
Unsloth: Saving meta-llama-3.1-8b-q4_k_m-chandlerBingHarry-GGUF/pytorch_model-00002-of-00004.bin...
Unsloth: Saving meta-llama-3.1-8b-q4_k_m-chandlerBingHarry-GGUF/pytorch_model-00003-of-00004.bin...
Unsloth: Saving meta-llama-3.1-8b-q4_k_m-chandlerBingHarry-GGUF/pytorch_model-00004-of-00004.bin...
Done.


Unsloth: Converting llama model. Can use fast conversion = False.


==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: CMAKE detected. Finalizing some steps for installation.
Unsloth: [1] Converting model at meta-llama-3.1-8b-q4_k_m-chandlerBingHarry-GGUF into f16 GGUF format.
The output location will be /content/meta-llama-3.1-8b-q4_k_m-chandlerBingHarry-GGUF/unsloth.F16.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: meta-llama-3.1-8b-q4_k_m-chandlerBingHarry-GGUF
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 

  0%|          | 0/1 [00:00<?, ?it/s]

unsloth.Q4_K_M.gguf:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Saved GGUF to https://huggingface.co/sgaeyl/meta-llama-3.1-8b-q4_k_m-chandlerBingHarry-GGUF
